# Notebook 3: LSTM-SNP with Fuzzy Gate Replacement

**Dataset**: Monthly Milk Production

## Description
This notebook implements **fuzzy gate replacement** for the LSTM-SNP model. The reset (r), 
consumption (c), and output (o) gates — which normally use hard sigmoid activation — are 
replaced with fuzzy inference systems. Each gate uses 2 Takagi-Sugeno rules with fixed Gaussian 
membership functions and trainable consequent parameters.

The generation gate (a) retains its original tanh activation. The rest of the SNP architecture 
is unchanged. Gradient clipping (norm=1.0) is applied for training stability.

In [1]:
# ============================================================
# ALL IMPORTS
# ============================================================

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras import Model
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from math import sqrt
import matplotlib.pyplot as plt

print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version: {np.__version__}")

TensorFlow version: 2.21.0
NumPy version: 2.5.1


In [2]:
# Default sigma value (will be overridden by the sigma loop)
CURRENT_SIGMA = 0.5

## Theory: Fuzzy Gate Replacement

### Modified Gate Computation
Instead of applying hard sigmoid directly, each gate (r, c, o) uses fuzzy inference:

**Standard**: $r(t) = \sigma(z_r)$ where $z_r = W_r x(t) + U_r u(t-1) + b_r$

**Fuzzy**: $r(t) = FuzzyInference(z_r, \bar{u}(t-1))$

The fuzzy inference uses:

**Fixed Gaussian Membership Functions**:
- $\mu_{low}(z) = \exp\left(-\frac{(z - (-1))^2}{2 \cdot 0.5^2}\right)$
- $\mu_{high}(z) = \exp\left(-\frac{(z - (+1))^2}{2 \cdot 0.5^2}\right)$

**2 Rules per gate** (trainable consequents):
1. IF $z_{gate}$ is low → $y_0 = a_0 z + b_0 \bar{u} + c_0$
2. IF $z_{gate}$ is high → $y_1 = a_1 z + b_1 \bar{u} + c_1$

**Output**: Gate value bounded via sigmoid to $(0, 1)$

### Unchanged
- Generation gate $a(t) = \tanh(\cdot)$ — unchanged
- State update: $u(t) = r \cdot u_{t-1} - c \cdot a$, $h(t) = o \cdot a$ — unchanged
- Gradient clipping (norm ≤ 1.0) applied for stability

### Fuzzy Gate LSTM-SNP Cell

In [3]:
import torch
import torch.nn as nn

class LSTMSNPCell(nn.Module):
    """Original LSTM-SNP cell — unchanged (hard_sigmoid gates)."""
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.W = nn.Linear(input_size, 4 * hidden_size, bias=True)
        self.U = nn.Linear(hidden_size, 4 * hidden_size, bias=True)
        nn.init.xavier_uniform_(self.W.weight)
        nn.init.orthogonal_(self.U.weight)

    def forward(self, x, u_prev):
        z = self.W(x) + self.U(u_prev)
        z0 = z[:, :self.hidden_size]
        z1 = z[:, self.hidden_size:2*self.hidden_size]
        z2 = z[:, 2*self.hidden_size:3*self.hidden_size]
        z3 = z[:, 3*self.hidden_size:]
        r = torch.nn.functional.hardsigmoid(z0)
        c = torch.nn.functional.hardsigmoid(z1)
        o = torch.nn.functional.hardsigmoid(z2)
        a = torch.tanh(z3)
        u = r * u_prev - c * a
        h = o * a
        return h, u


class FuzzyLSTMSNPCell(nn.Module):
    """
    LSTM-SNP Cell with fuzzy gate replacement.
    Gates r, c, o computed via 2-rule Takagi-Sugeno fuzzy inference,
    fixed Gaussian MFs (sigma read from global CURRENT_SIGMA at
    construction time, matching the Keras cell's convention),
    trainable consequents, sigmoid-bounded output.
    Gate a keeps tanh (unchanged).
    """
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.mu_low = -1.0
        self.mu_high = 1.0
        # Read from global, exactly like the Keras cell does —
        # CURRENT_SIGMA must be set before this class is instantiated.
        self.sigma = CURRENT_SIGMA

        self.W = nn.Linear(input_size, 4 * hidden_size, bias=True)
        self.U = nn.Linear(hidden_size, 4 * hidden_size, bias=True)
        nn.init.xavier_uniform_(self.W.weight)
        nn.init.orthogonal_(self.U.weight)

        self.fuzzy_params = nn.ParameterDict()
        for gate_name in ['r', 'c', 'o']:
            for rule_idx in range(2):
                self.fuzzy_params[f'{gate_name}_{rule_idx}_a'] = nn.Parameter(
                    torch.empty(hidden_size).uniform_(-0.1, 0.1))
                self.fuzzy_params[f'{gate_name}_{rule_idx}_b'] = nn.Parameter(
                    torch.empty(hidden_size).uniform_(-0.1, 0.1))
                self.fuzzy_params[f'{gate_name}_{rule_idx}_c'] = nn.Parameter(
                    torch.zeros(hidden_size))

    def _gaussian_mf(self, x, center):
        return torch.exp(-(x - center)**2 / (2.0 * self.sigma**2))

    def _fuzzy_gate(self, z_gate, u_mean, gate_name):
        w_low = self._gaussian_mf(z_gate, self.mu_low)
        w_high = self._gaussian_mf(z_gate, self.mu_high)

        a0 = self.fuzzy_params[f'{gate_name}_0_a']
        b0 = self.fuzzy_params[f'{gate_name}_0_b']
        c0 = self.fuzzy_params[f'{gate_name}_0_c']
        a1 = self.fuzzy_params[f'{gate_name}_1_a']
        b1 = self.fuzzy_params[f'{gate_name}_1_b']
        c1 = self.fuzzy_params[f'{gate_name}_1_c']

        y0 = a0 * z_gate + b0 * u_mean + c0
        y1 = a1 * z_gate + b1 * u_mean + c1

        numerator = w_low * y0 + w_high * y1
        denominator = w_low + w_high + 1e-8
        gate_raw = numerator / denominator

        return torch.sigmoid(gate_raw)

    def forward(self, x, u_prev):
        z = self.W(x) + self.U(u_prev)
        z0 = z[:, :self.hidden_size]
        z1 = z[:, self.hidden_size:2*self.hidden_size]
        z2 = z[:, 2*self.hidden_size:3*self.hidden_size]
        z3 = z[:, 3*self.hidden_size:]

        u_mean = u_prev.mean(dim=-1, keepdim=True).expand(-1, self.hidden_size)

        r = self._fuzzy_gate(z0, u_mean, 'r')
        c = self._fuzzy_gate(z1, u_mean, 'c')
        o = self._fuzzy_gate(z2, u_mean, 'o')
        a = torch.tanh(z3)

        u = r * u_prev - c * a
        h = o * a
        return h, u


class RNNModel(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.cell = FuzzyLSTMSNPCell(input_size, hidden_size)
        self.out = nn.Linear(hidden_size, 1)
        self.u = None

    def reset_states(self, batch_size, device):
        self.u = torch.zeros(batch_size, self.hidden_size, device=device)

    def detach_states(self):
        if self.u is not None:
            self.u = self.u.detach()

    def forward(self, x):
        if self.u is None or self.u.device != x.device or self.u.size(0) != x.size(0):
            self.reset_states(x.size(0), x.device)
        h, self.u = self.cell(x[:, 0, :], self.u)
        return self.out(h)


def build_model_torch(input_dim, units, batch_size=1):
    # batch_size accepted for call-signature compatibility with build_model_keras,
    # unused here — actual batch size is set dynamically via reset_states().
    return RNNModel(input_size=input_dim, hidden_size=units)


# ============================================================
# Quick model check
# ============================================================
CURRENT_SIGMA = 0.5  # must be set before FuzzyLSTMSNPCell() is constructed
model = build_model_torch(input_dim=1, units=8, batch_size=1)
print(model)
print(f"Total params: {sum(p.numel() for p in model.parameters())}")

RNNModel(
  (cell): FuzzyLSTMSNPCell(
    (W): Linear(in_features=1, out_features=32, bias=True)
    (U): Linear(in_features=8, out_features=32, bias=True)
    (fuzzy_params): ParameterDict(
        (r_0_a): Parameter containing: [torch.FloatTensor of size 8]
        (r_0_b): Parameter containing: [torch.FloatTensor of size 8]
        (r_0_c): Parameter containing: [torch.FloatTensor of size 8]
        (r_1_a): Parameter containing: [torch.FloatTensor of size 8]
        (r_1_b): Parameter containing: [torch.FloatTensor of size 8]
        (r_1_c): Parameter containing: [torch.FloatTensor of size 8]
        (c_0_a): Parameter containing: [torch.FloatTensor of size 8]
        (c_0_b): Parameter containing: [torch.FloatTensor of size 8]
        (c_0_c): Parameter containing: [torch.FloatTensor of size 8]
        (c_1_a): Parameter containing: [torch.FloatTensor of size 8]
        (c_1_b): Parameter containing: [torch.FloatTensor of size 8]
        (c_1_c): Parameter containing: [torch.Floa

## Data Pipeline — milk production

In [6]:
import os

In [7]:
# ============================================================
# 1. Load Time Series Data
# ============================================================
import os

DATA_PATH = r"C:\Users\paulp\OneDrive\Desktop\fuzzy_LSTM\dataset\monthly-milk-production-pounds-p.csv"

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Could not find the dataset at:\n  {DATA_PATH}\n"
        "Update DATA_PATH above to point at monthly-milk-production-pounds-p.csv "
        "on this machine (native Windows path, e.g. C:\\Users\\... -- "
        "/mnt/c/... paths only work inside WSL, not native Windows Python)."
    )

series = pd.read_csv(
    DATA_PATH,
    header=0,
    parse_dates=[0],
    index_col=0
)

raw_values = series.values.flatten()
print(f"Data shape: {raw_values.shape}")
print(f"First 5 values: {raw_values[:5]}")

Data shape: (168,)
First 5 values: [589 561 640 656 727]


In [8]:
# ============================================================
# 2. First-Order Differencing
# ============================================================

def difference(dataset, interval=1):
    diff = []
    for i in range(interval, len(dataset)):
        value = dataset[i] - dataset[i - interval]
        diff.append(value)
    return np.array(diff)

diff_values = difference(raw_values, 1)

In [9]:
# ============================================================
# 3. Convert to Supervised Learning Format (lag=1)
# ============================================================

def timeseries_to_supervised(data, lag=1):
    df = pd.DataFrame(data)
    columns = [df.shift(i) for i in range(1, lag+1)]
    columns.append(df)
    df = pd.concat(columns, axis=1)
    df.fillna(0, inplace=True)
    return df.values

supervised = timeseries_to_supervised(diff_values, 1)
print(f"Supervised data shape: {supervised.shape}")

Supervised data shape: (167, 2)


In [10]:
# ============================================================
# 4. Train-Validation-Test Split (Last 60 points as Test)
# ============================================================
n = len(supervised)
test_size = 60
train_val = supervised[:n - test_size]
test = supervised[n - test_size:]

val_size = int(len(train_val) * 0.1)
train = train_val[:-val_size]
val = train_val[-val_size:]

print(f"Train: {train.shape}, Val: {val.shape}, Test: {test.shape}")

# ============================================================
# 5. Feature Scaling
# ============================================================
scaler = MinMaxScaler(feature_range=(-1, 1))
scaler.fit(train)  # fit only on train to avoid leakage
train_scaled = scaler.transform(train)
val_scaled = scaler.transform(val)
test_scaled = scaler.transform(test)

Train: (97, 2), Val: (10, 2), Test: (60, 2)


In [11]:
# ============================================================
# 6. Reshape for RNN Input
# ============================================================

X_train, y_train = train_scaled[:, 0:-1], train_scaled[:, -1]
X_train = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))

X_test, y_test = test_scaled[:, 0:-1], test_scaled[:, -1]
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")

X_train shape: (97, 1, 1), y_train shape: (97,)


## Training Loop

In [12]:
import os

# ============================================================
# Multi-Sigma Experiment Protocol (PyTorch) -- with checkpointing
# σ values from Eq. (21) & (22): 0.25, 0.5, 0.75, 1.0
#
# Checkpointing: after every run, the trained model's state_dict + that
# run's metrics are written to disk under CHECKPOINT_DIR. If this cell is
# re-run (kernel restart, crash, "Restart & Run All", etc.), it picks up
# from the last completed run per sigma instead of retraining from scratch.
# ============================================================

CHECKPOINT_DIR = "checkpoints_fuzzy_gate_dowjones"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

def _ckpt_path(sigma, run):
    return os.path.join(CHECKPOINT_DIR, f"sigma_{sigma}_run_{run}.pt")

def _sigma_summary_path(sigma):
    return os.path.join(CHECKPOINT_DIR, f"sigma_{sigma}_summary.pt")

SIGMA_VALUES = [0.25, 0.5, 0.75, 1.0]
sigma_results = {}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n--- [PyTorch] RUNNING ON {device} ---\n")
print(f"Checkpoints will be read/written under: {os.path.abspath(CHECKPOINT_DIR)}\n")

for CURRENT_SIGMA in SIGMA_VALUES:
    print(f'\n{"="*60}')
    print(f'  SIGMA = {CURRENT_SIGMA}')
    print(f'{"="*60}')

    # If this sigma was already fully completed in a previous session, just
    # reload its saved summary (metrics + model state_dicts) and skip training.
    summary_path = _sigma_summary_path(CURRENT_SIGMA)
    if os.path.exists(summary_path):
        print(f"Found completed checkpoint for sigma={CURRENT_SIGMA} -> loading, skipping training.")
        saved = torch.load(summary_path, map_location=device, weights_only=False)

        all_models = []
        for sd in saved['model_state_dicts']:
            m = build_model_torch(input_dim=1, units=8, batch_size=1).to(device)
            m.load_state_dict(sd)
            m.eval()
            all_models.append(m)

        sigma_results[CURRENT_SIGMA] = {
            'rmse': saved['rmse'],
            'mse': saved['mse'],
            'nmse': saved['nmse'],
            'predictions': saved['predictions'],
            'models': all_models,
            'mean_rmse': saved['mean_rmse'],
            'std_rmse': saved['std_rmse'],
            'mean_mse': saved['mean_mse'],
            'std_mse': saved['std_mse'],
            'mean_nmse': saved['mean_nmse'],
            'std_nmse': saved['std_nmse'],
            'best_idx': saved['best_idx'],
            'best_rmse': saved['best_rmse'],
        }
        print(f"Loaded {len(all_models)} models for sigma={CURRENT_SIGMA} from checkpoint.")
        continue

    # ============================================================
    # 60-Run Experiment Protocol
    # ============================================================

    all_rmse = []
    all_mse = []
    all_nmse = []
    all_predictions = []
    all_losses = []
    all_models = []  # keep every trained model for later noise-robustness eval

    N_RUNS = 60
    for run in range(N_RUNS):

        # Resume mid-sigma: if this specific run was already checkpointed
        # (e.g. the kernel died partway through this sigma), reload it
        # instead of retraining.
        run_ckpt_path = _ckpt_path(CURRENT_SIGMA, run)
        if os.path.exists(run_ckpt_path):
            print(f'\n===== RUN {run+1}/{N_RUNS} (resumed from checkpoint) =====')
            ckpt = torch.load(run_ckpt_path, map_location=device, weights_only=False)

            model = build_model_torch(input_dim=1, units=8, batch_size=1).to(device)
            model.load_state_dict(ckpt['model_state_dict'])
            model.eval()

            all_losses.append(ckpt['run_losses'])
            all_models.append(model)
            all_rmse.append(ckpt['rmse'])
            all_mse.append(ckpt['mse'])
            all_nmse.append(ckpt['nmse'])
            all_predictions.append(ckpt['predictions'])
            print(f"Loaded run {run+1} -- RMSE: {ckpt['rmse']:.6f}, MSE: {ckpt['mse']:.6f}, NMSE: {ckpt['nmse']:.10f}")
            continue

        print(f'\n===== RUN {run+1}/{N_RUNS} =====')

        np.random.seed(run)
        torch.manual_seed(run)

        model = build_model_torch(input_dim=1, units=8, batch_size=1).to(device)

        # Initialize consumption gate bias to 1.0 (forget gate equivalent)
        with torch.no_grad():
            hs = model.hidden_size
            model.cell.U.bias.data[hs:2*hs] = 1.0

        optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
        criterion = nn.MSELoss()

        run_losses = []
        X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
        y_train_t = torch.tensor(y_train, dtype=torch.float32).to(device)
        n_samples = X_train_t.size(0)

        for epoch in range(100):
            model.train()
            model.reset_states(1, device)
            epoch_loss = 0.0
            for i in range(n_samples):
                x_i = X_train_t[i:i+1]
                y_i = y_train_t[i:i+1]

                optimizer.zero_grad()
                pred = model(x_i)
                loss = criterion(pred.squeeze(-1), y_i)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                model.detach_states()
                epoch_loss += loss.item()

            avg_loss = epoch_loss / n_samples
            run_losses.append(avg_loss)
            print(f"Epoch {epoch+1}/100 completed. Loss: {avg_loss:.6f}")

        all_losses.append(run_losses)
        model.reset_states(1, device)
        all_models.append(model)
        print(f'Training complete for run {run+1}')

        model.eval()
        with torch.no_grad():
            for i in range(len(train_scaled)):
                X_raw = train_scaled[i, 0:-1]
                X_input = torch.tensor(X_raw, dtype=torch.float32).view(1, 1, len(X_raw)).to(device)
                model(X_input)

        predictions = []
        model.eval()
        with torch.no_grad():
            for i in range(len(test_scaled)):
                X, y = test_scaled[i, 0:-1], test_scaled[i, -1]
                X_input = torch.tensor(X, dtype=torch.float32).view(1, 1, len(X)).to(device)
                yhat = model(X_input).item()

                new_row = [x for x in X] + [yhat]
                array = np.array(new_row).reshape(1, len(new_row))
                inverted = scaler.inverse_transform(array)[0, -1]
                inverted = inverted + raw_values[len(train) + i]
                predictions.append(inverted)

                expected = raw_values[len(train) + i + 1]
                print(f'Month={i+1}, Predicted={inverted:.4f}, Expected={expected:.4f}')

        actual = raw_values[-len(test_scaled):]
        rmse = sqrt(mean_squared_error(actual, predictions))
        mse = mean_squared_error(actual, predictions)
        meanV = np.mean(actual)
        dominator = np.linalg.norm(np.array(predictions) - meanV, 2)
        nmse = mse / np.power(dominator, 2)

        all_rmse.append(rmse)
        all_mse.append(mse)
        all_nmse.append(nmse)
        all_predictions.append(predictions)

        print(f'Run {run+1} -- RMSE: {rmse:.6f}, MSE: {mse:.6f}, NMSE: {nmse:.10f}')

        # Checkpoint this run to disk immediately, so a crash later in this
        # sigma (or in a later sigma) never loses this run's work.
        torch.save({
            'model_state_dict': model.state_dict(),
            'run_losses': run_losses,
            'rmse': rmse,
            'mse': mse,
            'nmse': nmse,
            'predictions': predictions,
        }, run_ckpt_path)

    sigma_results[CURRENT_SIGMA] = {
        'rmse': list(all_rmse),
        'mse': list(all_mse),
        'nmse': list(all_nmse),
        'predictions': list(all_predictions),
        'models': list(all_models),
        'mean_rmse': float(np.mean(all_rmse)),
        'std_rmse': float(np.std(all_rmse)),
        'mean_mse': float(np.mean(all_mse)),
        'std_mse': float(np.std(all_mse)),
        'mean_nmse': float(np.mean(all_nmse)),
        'std_nmse': float(np.std(all_nmse)),
        'best_idx': int(np.argmin(all_rmse)),
        'best_rmse': float(np.min(all_rmse)),
    }

    # Consolidated per-sigma checkpoint (metrics + all 60 state_dicts), so a
    # fully completed sigma never needs to be retrained even if the individual
    # per-run checkpoint files above are deleted/cleaned up later.
    torch.save({
        'rmse': sigma_results[CURRENT_SIGMA]['rmse'],
        'mse': sigma_results[CURRENT_SIGMA]['mse'],
        'nmse': sigma_results[CURRENT_SIGMA]['nmse'],
        'predictions': sigma_results[CURRENT_SIGMA]['predictions'],
        'model_state_dicts': [m.state_dict() for m in all_models],
        'mean_rmse': sigma_results[CURRENT_SIGMA]['mean_rmse'],
        'std_rmse': sigma_results[CURRENT_SIGMA]['std_rmse'],
        'mean_mse': sigma_results[CURRENT_SIGMA]['mean_mse'],
        'std_mse': sigma_results[CURRENT_SIGMA]['std_mse'],
        'mean_nmse': sigma_results[CURRENT_SIGMA]['mean_nmse'],
        'std_nmse': sigma_results[CURRENT_SIGMA]['std_nmse'],
        'best_idx': sigma_results[CURRENT_SIGMA]['best_idx'],
        'best_rmse': sigma_results[CURRENT_SIGMA]['best_rmse'],
    }, summary_path)

    print(f'\n--- σ={CURRENT_SIGMA} Summary ---')
    print(f'RMSE: {np.mean(all_rmse):.6f} ± {np.std(all_rmse):.6f}')
    print(f'MSE:  {np.mean(all_mse):.6f} ± {np.std(all_mse):.6f}')
    print(f'NMSE: {np.mean(all_nmse):.10f} ± {np.std(all_nmse):.10f}')


--- [PyTorch] RUNNING ON cuda ---

Checkpoints will be read/written under: c:\Users\paulp\OneDrive\Desktop\fuzzy_LSTM\with noise\type_3_sigmoid\checkpoints_fuzzy_gate_dowjones


  SIGMA = 0.25
Found completed checkpoint for sigma=0.25 -> loading, skipping training.
Loaded 60 models for sigma=0.25 from checkpoint.

  SIGMA = 0.5
Found completed checkpoint for sigma=0.5 -> loading, skipping training.
Loaded 60 models for sigma=0.5 from checkpoint.

  SIGMA = 0.75
Found completed checkpoint for sigma=0.75 -> loading, skipping training.
Loaded 60 models for sigma=0.75 from checkpoint.

  SIGMA = 1.0
Found completed checkpoint for sigma=1.0 -> loading, skipping training.
Loaded 60 models for sigma=1.0 from checkpoint.


## Theory: Fuzzy Gate Replacement

### Modified Gate Computation
Instead of applying hard sigmoid directly, each gate (r, c, o) uses fuzzy inference:

**Standard**: $r(t) = \sigma(z_r)$ where $z_r = W_r x(t) + U_r u(t-1) + b_r$

**Fuzzy**: $r(t) = FuzzyInference(z_r, \bar{u}(t-1))$

The fuzzy inference uses:

**Fixed Gaussian Membership Functions**:
- $\mu_{low}(z) = \exp\left(-\frac{(z - (-1))^2}{2 \cdot 0.5^2}\right)$
- $\mu_{high}(z) = \exp\left(-\frac{(z - (+1))^2}{2 \cdot 0.5^2}\right)$

**2 Rules per gate** (trainable consequents):
1. IF $z_{gate}$ is low → $y_0 = a_0 z + b_0 \bar{u} + c_0$
2. IF $z_{gate}$ is high → $y_1 = a_1 z + b_1 \bar{u} + c_1$

**Output**: Gate value bounded via sigmoid to $(0, 1)$

### Unchanged
- Generation gate $a(t) = \tanh(\cdot)$ — unchanged
- State update: $u(t) = r \cdot u_{t-1} - c \cdot a$, $h(t) = o \cdot a$ — unchanged
- Gradient clipping (norm ≤ 1.0) applied for stability

## Gaussian Noise-Robustness Sweep (0.5% / 5% / 10% / 15%)

Same protocol as the SNN-Transformer / LSTM-SNP noise-robustness notebooks:

`x_noisy(t) = x(t) + eps(t)`,  `eps(t) ~ N(0, sigma_eps^2)`,  `sigma_eps = eta * std(x)`

Noise is injected into the **input features only** (never the targets), using the 60 models
already trained for each sigma above (`sigma_results[sigma]['models']`) -- no retraining is
needed. Each (sigma, noise level) combination is evaluated over multiple noise draws and
reported as a mean +/- std over the 60 runs, exactly mirroring the clean-data summary.

In [14]:
# ============================================================
# Gaussian noise injection (same protocol as the SNN-Transformer /
# LSTM-SNP noise-robustness notebooks)
# ============================================================
ref_std = X_train.std()
print(f"Reference std(x) used for noise scaling (train inputs): {ref_std:.6f}")

NOISE_LEVELS = [0.0, 0.005, 0.05, 0.10, 0.15]  # 0%, 0.5%, 5%, 10%, 15%
N_NOISE_DRAWS = 10  # average over multiple eps(t) realizations per model, per level

def add_gaussian_noise(X, noise_level, ref_std, seed=None):
    # noise_level == 0.0 -> sigma_eps == 0 -> eps is all zeros, so this
    # naturally reduces to the clean (no-noise) case with no special-casing.
    """
    x_noisy(t) = x(t) + eps(t),  eps(t) ~ N(0, sigma_eps^2),  sigma_eps = noise_level * std(x)
    Applied to input features X only -- never to targets/labels.
    """
    rng = np.random.default_rng(seed)
    sigma_eps = noise_level * ref_std
    eps = rng.normal(loc=0.0, scale=sigma_eps, size=X.shape)
    return X + eps

Reference std(x) used for noise scaling (train inputs): 0.524874


In [15]:
# ============================================================
# Evaluation helper -- warms the recurrent state up on clean training data,
# then predicts on X_eval (clean or Gaussian-noise-corrupted test inputs).
# Test targets are always the clean, ground-truth values. NMSE denominator
# uses the actual series (kept consistent with the SNN-Transformer noise
# notebook so results are directly comparable across variants).
# ============================================================
from scipy import stats

def evaluate_on_test(model, X_eval, train, train_scaled, raw_values, scaler, device):
    model.eval()
    model.reset_states(1, device)

    with torch.no_grad():
        for i in range(len(train_scaled)):
            X_raw = train_scaled[i, 0:-1]
            X_input = torch.tensor(X_raw, dtype=torch.float32).view(1, 1, len(X_raw)).to(device)
            model(X_input)

        predictions = []
        for i in range(len(X_eval)):
            X = X_eval[i]
            X_input = torch.tensor(X, dtype=torch.float32).view(1, 1, len(X)).to(device)
            yhat = model(X_input).item()

            new_row = [x for x in X] + [yhat]
            array = np.array(new_row).reshape(1, len(new_row))
            inverted = scaler.inverse_transform(array)[0, -1]
            inverted = inverted + raw_values[len(train) + i]
            predictions.append(inverted)

    actual = raw_values[-len(X_eval):]
    mse = mean_squared_error(actual, predictions)
    rmse = sqrt(mse)
    meanV = np.mean(actual)
    denominator = np.linalg.norm(np.array(actual) - meanV, 2)
    nmse = mse / np.power(denominator, 2)
    return predictions, rmse, mse, nmse

In [ ]:
## Results

In [16]:
# ============================================================
# Noise-robustness sweep, run for every sigma trained above
# (averaged over multiple noise draws per model)
# ============================================================

# Guard: make sure the training-loop cell above actually saved the trained
# models before we try to reuse them here. If sigma_results was populated by
# an older run of the training cell (before model-saving was added), or the
# kernel was restarted and only some cells were re-run, 'models' will be
# missing -- fail with a clear message instead of a bare KeyError.
missing = [s for s in SIGMA_VALUES if s not in sigma_results or 'models' not in sigma_results[s]]
if missing:
    raise RuntimeError(
        f"sigma_results is missing trained models for sigma(s): {missing}. "
        "Re-run the TRAINING LOOP cell (the one that builds sigma_results) from a "
        "fresh kernel -- it must run to completion for every sigma in SIGMA_VALUES "
        "before this noise sweep can reuse the trained models."
    )

noise_robustness_by_sigma = {}  # sigma -> {noise_level -> {'rmse':[...], 'mse':[...], 'nmse':[...]}}
clean_by_sigma = {}             # sigma -> {'rmse':[...], 'mse':[...], 'nmse':[...]} (re-evaluated with evaluate_on_test)

for sigma_val in SIGMA_VALUES:
    print(f'\n{"="*60}')
    print(f'  NOISE SWEEP -- SIGMA = {sigma_val}')
    print(f'{"="*60}')

    models = sigma_results[sigma_val]['models']

    # Re-evaluate each model on clean X_test with evaluate_on_test so the clean
    # baseline uses the exact same evaluation path as the noisy conditions
    # (needed for a fair paired comparison -- the original training-loop RMSE
    # used a slightly different NMSE denominator, see the cell above).
    clean_rmse, clean_mse, clean_nmse = [], [], []
    for model in models:
        _, rmse, mse, nmse = evaluate_on_test(model, X_test, train, train_scaled, raw_values, scaler, device)
        clean_rmse.append(rmse)
        clean_mse.append(mse)
        clean_nmse.append(nmse)
    clean_by_sigma[sigma_val] = {'rmse': clean_rmse, 'mse': clean_mse, 'nmse': clean_nmse}

    noise_robustness_by_sigma[sigma_val] = {}
    for noise_level in NOISE_LEVELS:
        level_rmse, level_mse, level_nmse = [], [], []

        for run, model in enumerate(models):
            draw_rmse, draw_mse, draw_nmse = [], [], []
            for draw in range(N_NOISE_DRAWS):
                seed = hash((sigma_val, noise_level, run, draw)) % (2 ** 32)
                X_test_noisy = add_gaussian_noise(X_test, noise_level, ref_std, seed=seed)
                _, rmse, mse, nmse = evaluate_on_test(
                    model, X_test_noisy, train, train_scaled, raw_values, scaler, device
                )
                draw_rmse.append(rmse)
                draw_mse.append(mse)
                draw_nmse.append(nmse)

            level_rmse.append(np.mean(draw_rmse))  # this model's average over noise draws
            level_mse.append(np.mean(draw_mse))
            level_nmse.append(np.mean(draw_nmse))

        noise_robustness_by_sigma[sigma_val][noise_level] = {
            'rmse': level_rmse, 'mse': level_mse, 'nmse': level_nmse
        }
        label = "no noise" if noise_level == 0.0 else f"{noise_level * 100:5.1f}% noise"
        print(f"[{label:>10}]  "
              f"RMSE: {np.mean(level_rmse):.6f} +/- {np.std(level_rmse):.6f}  |  "
              f"MSE: {np.mean(level_mse):.6f} +/- {np.std(level_mse):.6f}  |  "
              f"NMSE: {np.mean(level_nmse):.10f} +/- {np.std(level_nmse):.10f}")


  NOISE SWEEP -- SIGMA = 0.25
[  no noise]  RMSE: 58.974476 +/- 1.014653  |  MSE: 3479.018334 +/- 121.981602  |  NMSE: 0.0163435337 +/- 0.0005730382
[  0.5% noise]  RMSE: 58.974386 +/- 1.014937  |  MSE: 3479.008295 +/- 122.018774  |  NMSE: 0.0163434866 +/- 0.0005732128
[  5.0% noise]  RMSE: 58.977964 +/- 1.017604  |  MSE: 3479.438545 +/- 122.406439  |  NMSE: 0.0163455078 +/- 0.0005750340
[ 10.0% noise]  RMSE: 58.986856 +/- 1.014905  |  MSE: 3480.488723 +/- 122.026785  |  NMSE: 0.0163504412 +/- 0.0005732505
[ 15.0% noise]  RMSE: 58.992862 +/- 1.024877  |  MSE: 3481.229782 +/- 123.412593  |  NMSE: 0.0163539225 +/- 0.0005797606

  NOISE SWEEP -- SIGMA = 0.5
[  no noise]  RMSE: 58.833324 +/- 0.973932  |  MSE: 3462.308591 +/- 117.233033  |  NMSE: 0.0162650357 +/- 0.0005507306
[  0.5% noise]  RMSE: 58.833438 +/- 0.974777  |  MSE: 3462.323581 +/- 117.344881  |  NMSE: 0.0162651061 +/- 0.0005512561
[  5.0% noise]  RMSE: 58.832701 +/- 0.977014  |  MSE: 3462.242379 +/- 117.672461  |  NMSE: 0.016

## Observations

### Fuzzy Gate Replacement on Monthly Milk Production

**Run the notebook to generate results and fill in observations:**

1. **Prediction Quality**: Compare RMSE/MSE/NMSE with other variants
2. **Training Stability**: Examine loss curves for convergence behavior
3. **Prediction Tracking**: Assess how well predictions track actual values
4. **Computational Cost**: Note training time per run

*After running all 5 variant notebooks, perform cross-variant comparison to evaluate 
whether fuzzy logic improves nonlinearity handling, interpretability, and prediction performance.*

In [ ]:
# ============================================================
# NOTEBOOK TIMER — END
# ============================================================
_NOTEBOOK_END_TIME = _timer_module.time()
_NOTEBOOK_ELAPSED = _NOTEBOOK_END_TIME - _NOTEBOOK_START_TIME
_hours, _rem = divmod(_NOTEBOOK_ELAPSED, 3600)
_minutes, _seconds = divmod(_rem, 60)
print(f"\nTotal notebook execution time: {int(_hours)}h {int(_minutes)}m {_seconds:.2f}s")
print(f"Total seconds: {_NOTEBOOK_ELAPSED:.2f}")
